# Check GPU + install packages

In [ ]:
!nvidia-smi      # confirm GPU is ready
!pip install -q -U sentence-transformers "psycopg[binary]"


Sun Jun 21 12:02:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Connect to Neon

In [ ]:
import getpass, psycopg

# Paste connection string into input
NEON_URL = getpass.getpass("Nenon connection string: ")
conn = psycopg.connect(NEON_URL)
conn.autocommit = True 
print("Connected to Neon success")

Connected to Neon success


# Read jobs dont have embedding

Length of rows must to be equal amount jobs in `fact_job_postings` because in first time run the embedding table is empty

In [6]:
SELECT_SQL = """
    SELECT f.job_id, f.job_title, f.job_description, f.job_requirement
    FROM warehouse_warehouse.fact_job_postings AS f
    LEFT JOIN warehouse_warehouse.job_embeddings AS e USING (job_id)
    WHERE e.job_id IS NULL
"""

with conn.cursor() as cur:
    cur.execute(SELECT_SQL)
    rows = cur.fetchall()
print(f"{len(rows)} need embedded")

814 need embedded


# Load `bge-m3` to GPU

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3", device="cuda")
model.max_seq_length = 1024
print("Mode loaded on", model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Mode loaded on cuda:0


# Merge text -> encode -> UPSERT into NeonDB

In [10]:
def build_text(row):
    _, title, desc, req = row
    return "\n".join(p for p in (title, desc, req) if p)

job_ids = [r[0] for r in rows]
texts = [build_text(r) for r in rows]

embeddings = model.encode(
    texts,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
)

UPSERT_SQL = """
    INSERT INTO warehouse_warehouse.job_embeddings (job_id, embedding, model_name)
    VALUES (%s, %s::vector, %s)
    ON CONFLICT (job_id) DO UPDATE
        SET embedding = EXCLUDED.embedding,
        model_name = EXCLUDED.model_name,
        embedded_at = NOW()
"""

params = [(job_id, str(vec.tolist()), "BAAI/bge-m3")
          for job_id, vec in zip(job_ids, embeddings)]

import psycopg
CHUNK = 500
with psycopg.connect(NEON_URL) as conn2:           # Open again connection to neondb
    with conn2.cursor() as cur:
        for i in range(0, len(params), CHUNK):
            cur.executemany(UPSERT_SQL, params[i:i+CHUNK])
            conn2.commit()                          # commit từng chunk → resume được nếu rớt
            print(f"  Write {min(i+CHUNK, len(params))}/{len(params)}")
print("UPSERT done")

Batches:   0%|          | 0/102 [00:00<?, ?it/s]

  Write 500/814
  Write 814/814
UPSERT done


In [11]:
import psycopg
with psycopg.connect(NEON_URL) as conn3:
    with conn3.cursor() as cur:
        cur.execute("""SELECT count(*), min(vector_dims(embedding)), max(embedded_at)
                       FROM warehouse_warehouse.job_embeddings""")
        print(cur.fetchone())     # (n, 1024, last_run)


(814, 1024, datetime.datetime(2026, 6, 21, 12, 58, 20, 253393))
